Conectar ao banco de dados

Consultar os dados via sql ( Carregar via pandas dataframe)

Gravar os dados em um arquivo (Json, Csv, parquet)
    Se salvar em parquet, talvez seria necessário atualizar os datatypes do dataframe para que seja compatível com o parquet

Para essa aula, a atividade necessita de parquet como output no bucket

```text
BRONZE/fornecedores_legado/
└── ano=2026/
    └── mes=11/
        └── dia=16/
            └── execucao=20261116T200000Z/
                ├── fornecedores_legado.parquet
                ├── _manifest.json
                └── _SUCCESS
```

## Configuração e conexão ao Banco de Dados

In [2]:
import os
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv

# Usa o .env da raiz; no Docker, preserva as variáveis do container.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".env").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Arquivo .env não encontrado na pasta do projeto.")
load_dotenv(PROJECT_ROOT / ".env", override=False)


def env(nome):
    valor = os.environ.get(nome, "").strip()
    if not valor:
        raise ValueError(f"Defina {nome} no .env ou no ambiente.")
    return valor


def conectar_postgres():
    return psycopg2.connect(
        host=env("POSTGRES_HOST"),
        port=int(env("POSTGRES_PORT")),
        dbname=env("POSTGRES_DB"),
        user=env("POSTGRES_USER"),
        password=env("POSTGRES_PASSWORD"),
        client_encoding="UTF8",
        application_name="02_batch_full_loader",
        connect_timeout=10,
    )


# Fecha a conexão anterior caso esta célula seja executada novamente.
if "conn" in globals() and not conn.closed:
    conn.close()

conn = conectar_postgres()
conn.autocommit = True

## Teste da query

In [4]:
# Lista as tabelas do schema public, definido em sql/02_schema.sql.
try:
    with conn.cursor() as cur:
        cur.execute("""
            select
            id_fornecedor
            ,razao_social
            ,cnpj
            ,status_credito
            ,limite_credito 
        from fornecedores_legado limit 15
        """)
        teste = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
except Exception:
    conn.close()
    raise

print("Conexão com o PostgreSQL estabelecida. Use conn nas próximas consultas.")
# Ao terminar a atividade, execute conn.close() para encerrar a conexão.
teste.head(5)

Conexão com o PostgreSQL estabelecida. Use conn nas próximas consultas.


,id_fornecedor,razao_social,cnpj,status_credito,limite_credito
0,FORN-BDD640FB06,Alves e Filhos Ltda.,39250467000125,EM_ANALISE,174344.14
1,FORN-1A3D1FA7BC,Andrade S/A,13846092000177,BLOQUEADO,375989.28
2,FORN-0822E8F36C,Câmara - ME,94061325000128,APROVADO,72877.81
3,FORN-9A1DE64481,Cunha Teixeira Ltda. S/A,16928450000189,APROVADO,386645.18
4,FORN-B38A088CA6,Nascimento S/A,61495723000101,BLOQUEADO,291534.98


# Consultar dados

In [7]:
try:
    with conn.cursor() as cur:
        cur.execute("""
            select
                id_fornecedor
                ,razao_social
                ,cnpj
                ,status_credito
                ,limite_credito 
            from fornecedores_legado
        """)
        df_fornecedor = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
except Exception:
    conn.close()
    raise


In [9]:
df_salvar_parquet = df_fornecedor.copy()

## Salvar os parquets no bucket

In [14]:
import io
import pyarrow.fs as fs
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc)

buffer = io.BytesIO()
df_salvar_parquet.to_parquet(buffer, engine="pyarrow", index=False)

minio = fs.S3FileSystem(
    access_key=env("MINIO_ROOT_USER"),
    secret_key=env("MINIO_ROOT_PASSWORD"),
    endpoint_override=env("S3_ENDPOINT_URL"),
    region="us-east-1",
)


ano = timestamp.strftime("%Y")
mes = timestamp.strftime("%m")
dia = timestamp.strftime("%d")
execucao = timestamp.strftime("%Y%m%dT%H%M%SZ")

caminho = (
    f"{env('MINIO_BUCKET_BRONZE')}/fornecedores_legado/ano={ano}/mes={mes}/dia={dia}/execucao={execucao}/"
    "fornecedores_legado.parquet"
)

with minio.open_output_stream(caminho) as destino:
    destino.write(buffer.getvalue())

print(f"Arquivo salvo em s3://{caminho}")

Arquivo salvo em s3://bronze/fornecedores_legado/ano=2026/mes=11/dia=17/execucao=20261117T001641Z/fornecedores_legado.parquet
